# SURD AGN starter notebook

This notebook collects the main steps from the SURD Colab setup and toy AGN reverberation-mapping test.

It includes:
- import/setup notes
- a synthetic AGN-like toy model
- a single-lag SURD run
- a lag scan
- a cleaner wrapper that collects results for all targets
- an analytic benchmark example

## Notes
- This notebook assumes the SURD repository has already been cloned.
- It uses the **core SURD modules only** and avoids `transport_map.py`, because that optional path needs `mpart` / MParT.
- The SURD `run()` helper plots all targets, but returns only the final target's decomposition. To fix that for analysis, this notebook defines `run_collect()`.
- The lag `nlag` is in **samples**, not physical days.


## 1. Environment and imports

Adjust `SURD_UTILS_PATH` if your clone is in a different directory.

In [ ]:
import sys
from pathlib import Path
from typing import Dict, Any

import numpy as np
import matplotlib.pyplot as plt

# Change this path if needed
SURD_UTILS_PATH = Path('/content/SURD/utils')

if not SURD_UTILS_PATH.exists():
    raise FileNotFoundError(
        f'Could not find SURD utils directory at {SURD_UTILS_PATH}. '
        'Update SURD_UTILS_PATH to match your local or Colab clone.'
    )

sys.path.append(str(SURD_UTILS_PATH))

import surd
import it_tools
import analytic_eqs

print('Core SURD imports successful')


## 2. Helper functions

`run_collect()` is a small wrapper around the repo code so that results are returned for **all** targets, not only the final target in the loop.

In [ ]:
def zscore(x: np.ndarray) -> np.ndarray:
    """Standardise an array to zero mean and unit variance."""
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / np.std(x)


def run_collect(
    X: np.ndarray,
    nvars: int,
    nlag: int,
    nbins: int,
    axs: np.ndarray | None = None,
    print_results: bool = False,
) -> Dict[int, Dict[str, Any]]:
    """
    Run SURD for each target signal and return a clean dictionary of results.

    Parameters
    ----------
    X : np.ndarray
        Array of shape (nvars, ntimes). Each row is one signal.
    nvars : int
        Number of signals.
    nlag : int
        Lag in samples.
    nbins : int
        Number of histogram bins used by np.histogramdd.
    axs : np.ndarray or None
        Optional axes array of shape (nvars, 2) for SURD plotting.
    print_results : bool
        If True, print the SURD decomposition for each target.

    Returns
    -------
    dict
        Dictionary keyed by target index, with values containing:
            - 'I_R'
            - 'I_S'
            - 'MI'
            - 'info_leak'
    """
    results = {}

    for i in range(nvars):
        # Future of target i versus present/past of all variables
        Y = np.vstack([X[i, nlag:], X[:, :-nlag]])

        # Multi-dimensional histogram approximation to the joint distribution
        hist, _ = np.histogramdd(Y.T, nbins)

        # SURD decomposition
        I_R, I_S, MI, info_leak = surd.surd(hist)

        results[i] = {
            'I_R': I_R,
            'I_S': I_S,
            'MI': MI,
            'info_leak': info_leak,
        }

        if print_results:
            print(f'\nSURD CAUSALITY FOR TARGET SIGNAL {i + 1}')
            surd.nice_print(I_R, I_S, MI, info_leak)

        if axs is not None:
            surd.plot(I_R, I_S, info_leak, axs[i, :], nvars, threshold=-0.01)

    return results


## 3. Synthetic AGN-like toy data

This creates three signals:
1. optical continuum: noisy proxy for a hidden UV driver
2. X-ray continuum: noisy proxy for the same hidden UV driver
3. Hβ line: delayed response to the UV driver

The `random_walk_driver` option is useful because it shows why the lag signal can look broad rather than sharply peaked.

In [ ]:
def make_synthetic_agn(
    N: int = 3000,
    tau: int = 8,
    seed: int = 42,
    random_walk_driver: bool = True,
) -> np.ndarray:
    """
    Build a synthetic AGN-like system with three signals.
    """
    rng = np.random.default_rng(seed)

    # Hidden UV driver
    if random_walk_driver:
        uv = np.cumsum(rng.normal(0, 0.15, N))
    else:
        uv = rng.normal(0, 1.0, N)
    uv = zscore(uv)

    # Two observed continuum proxies
    optical = 0.9 * uv + 0.25 * rng.normal(size=N)
    xray = 0.7 * uv + 0.35 * rng.normal(size=N)

    # Delayed broad-line response
    hbeta = np.zeros(N)
    hbeta[tau:] = 0.8 * uv[:-tau] + 0.25 * rng.normal(size=N - tau)
    hbeta[:tau] = 0.25 * rng.normal(size=tau)

    optical = zscore(optical)
    xray = zscore(xray)
    hbeta = zscore(hbeta)

    return np.vstack([optical, xray, hbeta])


## 4. Single-lag SURD run

This reproduces the first working toy example.

In [ ]:
tau = 8
X = make_synthetic_agn(N=3000, tau=tau, seed=42, random_walk_driver=True)
nvars = X.shape[0]

fig, axs = plt.subplots(nvars, 2, figsize=(12, 4 * nvars), squeeze=False)

# This is the repo's built-in wrapper.
# It prints and plots all targets, but returns only the final target result.
surd.run(X=X, nvars=nvars, nlag=tau, nbins=8, axs=axs)

plt.tight_layout()
plt.show()


## 5. Lag scan for target Signal 3

Here we use `run_collect()` so we can explicitly select the results for target Signal 3 (Hβ).

In [ ]:
tau_true = 8
X = make_synthetic_agn(N=3000, tau=tau_true, seed=42, random_walk_driver=True)
nvars = X.shape[0]

lags = np.arange(1, 21)
mi1, mi2, syn12, leak = [], [], [], []

for lag in lags:
    results = run_collect(X=X, nvars=nvars, nlag=lag, nbins=8)
    res = results[2]   # target index 2 = Signal 3 in human counting

    mi1.append(res['MI'].get((1,), np.nan))
    mi2.append(res['MI'].get((2,), np.nan))
    syn12.append(res['I_S'].get((1, 2), np.nan))
    leak.append(res['info_leak'])

plt.figure(figsize=(8, 5))
plt.plot(lags, mi1, marker='o', label='MI(1)')
plt.plot(lags, mi2, marker='o', label='MI(2)')
plt.plot(lags, syn12, marker='o', label='Synergy(1,2)')
plt.axvline(tau_true, linestyle='--', label='True lag')
plt.xlabel('Lag (samples)')
plt.ylabel('Information')
plt.title('Lag scan for target Signal 3 (Hβ)')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(lags, leak, marker='o')
plt.axvline(tau_true, linestyle='--')
plt.xlabel('Lag (samples)')
plt.ylabel('Information leak')
plt.title('Leak vs lag for target Signal 3 (Hβ)')
plt.tight_layout()
plt.show()

print('Interpretation note:')
print('If you use a random-walk hidden driver, the information curves often look broad rather than sharply peaked, because the driver has strong temporal autocorrelation.')


## 6. Compare random-walk and white-noise hidden drivers

This is useful for understanding why the lag curves can look broad in systems with long memory.

In [ ]:
tau_true = 8
lags = np.arange(1, 21)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), squeeze=False)

for ax, use_rw, title in zip(
    axs[0],
    [True, False],
    ['Random-walk driver', 'White-noise driver'],
    strict=True,
):
    X = make_synthetic_agn(
        N=3000,
        tau=tau_true,
        seed=42,
        random_walk_driver=use_rw,
    )

    leak = []
    for lag in lags:
        results = run_collect(X=X, nvars=3, nlag=lag, nbins=8)
        leak.append(results[2]['info_leak'])

    ax.plot(lags, leak, marker='o')
    ax.axvline(tau_true, linestyle='--')
    ax.set_xlabel('Lag (samples)')
    ax.set_ylabel('Information leak')
    ax.set_title(title)

plt.tight_layout()
plt.show()


## 7. Built-in analytic benchmark

The repo also contains synthetic benchmark systems in `analytic_eqs`.

Available motifs discovered in the session:
- `confounder`
- `mediator`
- `redundant_collider`
- `synergistic_collider`


In [ ]:
print([x for x in dir(analytic_eqs) if not x.startswith('_')])


In [ ]:
q1, q2, q3 = analytic_eqs.confounder(3000)
X = np.vstack([q1, q2, q3])

fig, axs = plt.subplots(3, 2, figsize=(10, 12), squeeze=False)
surd.run(X=X, nvars=3, nlag=1, nbins=8, axs=axs)
plt.tight_layout()
plt.show()


## 8. Suggested next steps

Once this notebook is working, the next scientific steps are:

1. Build more controlled toy cases:
   - one true driver
   - redundant proxies
   - genuinely synergistic drivers
2. Move to real AGN light curves:
   - resample to a uniform cadence
   - standardise each band
   - stack them into `X = np.vstack([...])`
   - scan over lags
3. Focus on the target row for the emission-line signal and compare:
   - unique information
   - redundancy
   - synergy
   - information leak
